The following function computes $s_n[s_\mu]$ using the recursive formula:
## $$n h_n[s_\mu] = \sum_{k=1}^n p_k[s_\mu] h_{n-k}[s_\mu]$$

In [1]:
from sage.all import *

In [1]:
#the following is the monomial expansion of s_mu in d-variables
@cached_function
def mc_s_mu(mu,d):
    s = SymmetricFunctions(QQ).s()
    return list(s(mu).expand(d).dict().items())

# defaultdict is useful for adding values adict[key] += c means that if key is already a key in adict
# it will add c to the current value, if not, it makes c the initial value.
from collections import defaultdict

# this is a function that computes a dictionary for the terms in s_n[s_mu]
# do it for d variables + an extra variable for the z
# this has worked really well for computing s_n[s_m] for a fixed m and for enn=0,1,2,3,...
# because I want the output in x1,x2,...,xd plus an extra variable z the length
# of the tuples are going to be d+1
def is_weakly_decreasing(tup):
    """
    test if a tuple is weakly decreasing
    """
    return all(tup[i]>=tup[i+1] for i in range(len(tup)-1))
@cached_function
def S_d(d):
    """
    The symmetric group with a sign
    """
    return [(tuple(v-1 for v in p),p.sign()) for p in Permutations(d)]
def sn_smu(enn, mu, d):
    """
    compute the terms in s_n[s_mu] of length at most d by calling hn_smu
    (this is a wrapper function so that lists are accepted in a cached function)
    """
    return hn_smu(enn, Partition(mu), d)
@cached_function
def hn_smu(enn, mu, d):
    """
    compute h_n[s_mu] using the recursive formula
    n h_n[s_mu] = sum_{k=1}^n p_k[s_mu] h_{n-k}[s_mu]
    """
    if enn==0:
        return { (0,)*(d+1) : 1 }
    else:
        out = defaultdict(int)
        for k in range(1,enn+1):
            for (w,ccc) in mc_s_mu(mu,d):
                for (v,c) in hn_smu(enn-k,mu,d).items():
                    for (p,cc) in S_d(d):
                        wv = tuple(k*w[i]+v[p[i]]+i-p[i] for i in range(d))
                        if is_weakly_decreasing(wv):
                            out[wv]+=c*cc*ccc
        return dict({v+(0,):c//enn for (v,c) in out.items() if c!=0})

In [2]:
nvars = 3 # number of variables
mu = Partition([4])
BR = QQ[','.join('x'+str(i) for i in range(1,nvars+1))+',z'] # the polynomial ring QQ[x1,x2,..,xd,z]
BR.inject_variables()
et = sage.rings.polynomial.polydict.ETuple

Defining x1, x2, x3, z


In this cell we put what we think is the denominator of ${\mathbb B}_{\mu}(x_1,x_2, \ldots, x_d; z)$

```
(1-z**4*x1**6*x2**6)*(1-z*x1**2*x2**1)*(1-z*x1**3)
```

for B_4 1 variable: `(1 - x1**4*z)`

In [23]:
plist = [
    [4],
    [4, 4],
    [3, 1],
    [6, 6],
    [9,9,2],
    [12,12,4],
    [8,2,2], #good stopping point
    [10,3,3],
    [5,5,2]
]

conj_den = mul( 1- mul(BR('x%s'%(i+1))**p[i] for i in range(nvars))*z**(sum(p)//sum(mu)) for p in [q+[0]*nvars for q in plist])

# what I was trying for B_4 3 vars: (1 - x1**4*z) * (1 - x1**4*x2**4*z**2) * (1 - x1**3*x2*z)  * (1 - x1**6*x2**6*z**3)*(1-x1**4*x2**4*x3**4*z**3)*(1-x1**6*x2**4*x3**2*z**3)*(1-x1^8*x2^2*x3^2*z^3)

#(1 - x1**4*z) * (1 - x2**4*z) * (1 - x1**2*x2**2*z)**2 * (1 - x1**3*x2*z) * (1 - x1*x2**3*z) * (1 + x1**2*x2**2*z) 
#(1 - x1**4*z) * (1 - x1**2*x2**2*z)**2 * (1 - x1**3*x2*z) * (1 + x1**2*x2**2*z)#(1 - x1**4*z) * (1 - x1**2*x2**2*z) 

In [25]:
@cached_function
def conj_den_4():
    return BR(expand(
        conj_den
        )).dict().items()#
@cached_function
def den_coeff(d):
    return BR({ et(list(t)[:-1]+[0]) : c for (t,c) in conj_den_4() if list(t)[-1]==d})
def calc_num(d):
    return sum(den_coeff(d-r)*BR(hn_smu(r,mu,nvars)) for r in range(d+1))
def diff_tup(p,q):
    return tuple([a-b for (a,b) in zip(q,p)])

In [ ]:
out={}
for d in range(0,100):
    CC = calc_num(d)
    #CCC = sorted(list(CC),key=lambda m: list(list(BR(m[1]).dict().items())[0][0])[::-1],reverse=True)
    CCC=list(CC)
    if CC:
        #out[d]=CCC[0][1].leading_item()[0]
        out[d]=tuple(list(CCC[0][1].dict().items())[0][0])
        print(d,out[d], len(list(CC)),CCC[0])
        delta = 1
        if d-delta in out:
            print(diff_tup(out[d-delta],out[d]))
        print("*************")
    else:
        print(d, "####" )

In [ ]:
final_numerator = 0

for d in range(0, 100):
    CC = calc_num(d)
    if CC:
        final_numerator += CC * (z**d)
        print(d)
    else:
        break

print(factor(final_numerator))

In [ ]:
final_numerator = 0
out = {}
for d in range(0, 100):
    CC = calc_num(d)
    if CC:
        CCC = list(CC)
        out[d] = tuple(list(CCC[0][1].dict().items())[0][0])
        final_numerator += CC * (z**d)
        if len(CCC) > 6:
            front = CCC[:3]
            back = CCC[-3:]
            print(d, out[d], len(CCC), "FRONT:", front, "BACK:", back)
        else:
            print(d, out[d], len(CCC), CCC)
            
        delta = 1
        if d - delta in out:
            print(diff_tup(out[d - delta], out[d]))
        print("*************")
    else:
        print(d, "####")

0 (0, 0, 0, 0) 1 [(1, 1)]
*************
1 (3, 1, 0, 0) 1 [(-1, x1^3*x2)]
(3, 1, 0, 0)
*************
2 (6, 2, 0, 0) 1 [(1, x1^6*x2^2)]
(3, 1, 0, 0)
*************
3 (7, 4, 1, 0) 4 [(1, x1^7*x2^4*x3), (1, x1^6*x2^4*x3^2), (-1, x1^5*x2^5*x3^2), (1, x1^4*x2^4*x3^4)]
(1, 2, 1, 0)
*************
4 (9, 6, 1, 0) 10 FRONT: [(1, x1^9*x2^6*x3), (1, x1^11*x2^3*x3^2), (1, x1^10*x2^4*x3^2)] BACK: [(1, x1^8*x2^4*x3^4), (-1, x1^7*x2^5*x3^4), (1, x1^6*x2^6*x3^4)]
(2, 2, 0, 0)
*************
5 (10, 9, 1, 0) 14 FRONT: [(1, x1^10*x2^9*x3), (-1, x1^14*x2^4*x3^2), (-1, x1^11*x2^7*x3^2)] BACK: [(1, x1^8*x2^8*x3^4), (1, x1^9*x2^6*x3^5), (2, x1^8*x2^6*x3^6)]
(1, 3, 0, 0)
*************
6 (13, 10, 1, 0) 19 FRONT: [(-1, x1^13*x2^10*x3), (-1, x1^14*x2^8*x3^2), (2, x1^12*x2^10*x3^2)] BACK: [(4, x1^10*x2^8*x3^6), (-1, x1^9*x2^9*x3^6), (2, x1^8*x2^8*x3^8)]
(3, 1, 0, 0)
*************
7 (16, 10, 2, 0) 30 FRONT: [(-1, x1^16*x2^10*x3^2), (-1, x1^15*x2^11*x3^2), (-1, x1^17*x2^8*x3^3)] BACK: [(3, x1^12*x2^8*x3^8), (-1, x1^11*

In [8]:
final_numerator.factor()

x1^6*x2^2*z^2 - x1^3*x2*z + 1

In [9]:
conj_den.factor()

(x1^2*x2^2*z + 1) * (x1^3*x2*z - 1) * (x1^4*z - 1) * (x1^2*x2^2*z - 1)^2 * (x1^4*x2^4*z^2 + x1^2*x2^2*z + 1)

In [10]:
final_numerator/conj_den

(x1^6*x2^2*z^2 - x1^3*x2*z + 1)/(x1^17*x2^11*z^7 - x1^14*x2^10*z^6 - x1^13*x2^11*z^6 - x1^13*x2^7*z^5 + x1^10*x2^10*z^5 - x1^11*x2^5*z^4 + x1^10*x2^6*z^4 + x1^9*x2^7*z^4 + x1^8*x2^4*z^3 + x1^7*x2^5*z^3 - x1^6*x2^6*z^3 + x1^7*x2*z^2 - x1^4*x2^4*z^2 - x1^4*z - x1^3*x2*z + 1)

In [18]:
# Initialize an empty list to hold the numerator polynomials for each degree
numerator_polys = []

# Assuming your code has a function get_coeff(d) or similar 
# that extracts the numerator term at degree d
for d in range(0, 27):
    term = calc_num(d) 
    # Multiply by z^d to build the formal power series
    numerator_polys.append(term * (z**d))

# Sum them up to get the full numerator polynomial
final_numerator = sum(numerator_polys)

# Now, factor the numerator to see its true structure
print((final_numerator))

x1^30*x2^22*x3^11*z^21 - x1^28*x2^21*x3^11*z^20 + x1^26*x2^20*x3^11*z^19 - x1^24*x2^19*x3^8*z^17 + x1^24*x2^18*x3^9*z^17 + x1^23*x2^19*x3^9*z^17 + 2*x1^22*x2^18*x3^8*z^16 + x1^23*x2^16*x3^9*z^16 - x1^21*x2^18*x3^9*z^16 + x1^22*x2^16*x3^7*z^15 - x1^20*x2^17*x3^8*z^15 + x1^20*x2^16*x3^6*z^14 + x1^21*x2^14*x3^7*z^14 + x1^20*x2^13*x3^9*z^14 - x1^19*x2^14*x3^9*z^14 + x1^18*x2^16*x3^5*z^13 + x1^20*x2^13*x3^6*z^13 - x1^18*x2^15*x3^6*z^13 - x1^17*x2^16*x3^6*z^13 - x1^18*x2^14*x3^7*z^13 - x1^18*x2^12*x3^9*z^13 + x1^18*x2^13*x3^5*z^12 - x1^16*x2^15*x3^5*z^12 - x1^18*x2^12*x3^6*z^12 - x1^17*x2^13*x3^6*z^12 + x1^15*x2^15*x3^6*z^12 + x1^18*x2^11*x3^7*z^12 - x1^16*x2^12*x3^8*z^12 + x1^17*x2^11*x3^5*z^11 - 2*x1^16*x2^12*x3^5*z^11 - x1^16*x2^10*x3^7*z^11 - x1^15*x2^11*x3^7*z^11 - x1^15*x2^11*x3^4*z^10 - x1^14*x2^12*x3^4*z^10 - 2*x1^14*x2^10*x3^6*z^10 + x1^13*x2^11*x3^6*z^10 - x1^14*x2^10*x3^3*z^9 + x1^12*x2^11*x3^4*z^9 + x1^15*x2^7*x3^5*z^9 - x1^13*x2^9*x3^5*z^9 - x1^12*x2^10*x3^5*z^9 - x1^14*x2^7*x3^

In [ ]:
factor(N)

Once you get a polynomial in the previous calculation, move on to giving the denominator:

In [10]:
out={}
nm_erator = 0
for d in range(0,20):
    CC = calc_num(d)
    nm_erator += z**d*CC
    print(z**d*CC)

-x2
2*x1*x2^2*x3*x4*z + x1^2*x2^2*z
-x1^2*x2^3*x3^2*x4^2*z^2 - 2*x1^3*x2^3*x3*x4*z^2 - x1^4*x2^3*z^2
x1^4*x2^4*x3^2*x4^2*z^3 + 2*x1^5*x2^4*x3*x4*z^3
-x1^6*x2^5*x3^2*x4^2*z^4 - x1^7*x2^4*x3^2*z^4 - x1^6*x2^5*x3^2*z^4 + x1^6*x2^4*x3^3*z^4 - x1^5*x2^5*x3^2*x4*z^4 - x1^6*x2^3*x3^2*x4^2*z^4 + x1*x2^2*x3*x4*z^4
2*x1^8*x2^5*x3^3*x4*z^5 + 2*x1^7*x2^6*x3^3*x4*z^5 - 2*x1^7*x2^5*x3^4*x4*z^5 + 2*x1^6*x2^6*x3^3*x4^2*z^5 + 2*x1^7*x2^4*x3^3*x4^3*z^5 + x1^9*x2^5*x3^2*z^5 - x1^7*x2^7*x3^2*z^5 - 2*x1^8*x2^5*x3^3*z^5 - x1^7*x2^5*x3^3*x4*z^5 - x1^6*x2^5*x3^4*x4*z^5 - x1^7*x2^5*x3^2*x4^2*z^5 - x1^6*x2^6*x3^2*x4^2*z^5 - x1^5*x2^5*x3^4*x4^2*z^5 - 2*x1^2*x2^3*x3^2*x4^2*z^5 - x1^3*x2^3*x3*x4*z^5
-x1^9*x2^6*x3^4*x4^2*z^6 - x1^8*x2^7*x3^4*x4^2*z^6 + x1^8*x2^6*x3^5*x4^2*z^6 - x1^7*x2^7*x3^4*x4^3*z^6 - x1^8*x2^5*x3^4*x4^4*z^6 - 2*x1^10*x2^6*x3^3*x4*z^6 + 2*x1^8*x2^8*x3^3*x4*z^6 + 4*x1^9*x2^6*x3^4*x4*z^6 + 2*x1^8*x2^6*x3^4*x4^2*z^6 + 2*x1^7*x2^6*x3^5*x4^2*z^6 + 2*x1^8*x2^6*x3^3*x4^3*z^6 + 2*x1^7*x2^7*x3^3*x4^3*z^6 

In [20]:
factor(nm_erator)

(-1) * x2 * (x1^26*x2^20*x3^11*z^19 - x1^24*x2^19*x3^8*z^17 + x1^24*x2^18*x3^9*z^17 + x1^23*x2^19*x3^9*z^17 + 2*x1^22*x2^18*x3^8*z^16 + x1^23*x2^16*x3^9*z^16 - x1^21*x2^18*x3^9*z^16 + x1^22*x2^16*x3^7*z^15 - x1^20*x2^17*x3^8*z^15 + x1^20*x2^16*x3^6*z^14 + x1^21*x2^14*x3^7*z^14 + x1^20*x2^13*x3^9*z^14 - x1^19*x2^14*x3^9*z^14 + x1^18*x2^16*x3^5*z^13 + x1^20*x2^13*x3^6*z^13 - x1^18*x2^15*x3^6*z^13 - x1^17*x2^16*x3^6*z^13 - x1^18*x2^14*x3^7*z^13 - x1^18*x2^12*x3^9*z^13 + x1^18*x2^13*x3^5*z^12 - x1^16*x2^15*x3^5*z^12 - x1^18*x2^12*x3^6*z^12 - x1^17*x2^13*x3^6*z^12 + x1^15*x2^15*x3^6*z^12 + x1^18*x2^11*x3^7*z^12 - x1^16*x2^12*x3^8*z^12 + x1^17*x2^11*x3^5*z^11 - 2*x1^16*x2^12*x3^5*z^11 - x1^16*x2^10*x3^7*z^11 - x1^15*x2^11*x3^7*z^11 - x1^15*x2^11*x3^4*z^10 - x1^14*x2^12*x3^4*z^10 - 2*x1^14*x2^10*x3^6*z^10 + x1^13*x2^11*x3^6*z^10 - x1^14*x2^10*x3^3*z^9 + x1^12*x2^11*x3^4*z^9 + x1^15*x2^7*x3^5*z^9 - x1^13*x2^9*x3^5*z^9 - x1^12*x2^10*x3^5*z^9 - x1^14*x2^7*x3^6*z^9 + x1^12*x2^9*x3^6*z^9 - x1^12*x

In [21]:
Bmu_d = nm_erator/conj_den

In [22]:
Bmu_d

(-x1^26*x2^20*x3^11*z^19 + x1^24*x2^19*x3^8*z^17 - x1^24*x2^18*x3^9*z^17 - x1^23*x2^19*x3^9*z^17 - 2*x1^22*x2^18*x3^8*z^16 - x1^23*x2^16*x3^9*z^16 + x1^21*x2^18*x3^9*z^16 - x1^22*x2^16*x3^7*z^15 + x1^20*x2^17*x3^8*z^15 - x1^20*x2^16*x3^6*z^14 - x1^21*x2^14*x3^7*z^14 - x1^20*x2^13*x3^9*z^14 + x1^19*x2^14*x3^9*z^14 - x1^18*x2^16*x3^5*z^13 - x1^20*x2^13*x3^6*z^13 + x1^18*x2^15*x3^6*z^13 + x1^17*x2^16*x3^6*z^13 + x1^18*x2^14*x3^7*z^13 + x1^18*x2^12*x3^9*z^13 - x1^18*x2^13*x3^5*z^12 + x1^16*x2^15*x3^5*z^12 + x1^18*x2^12*x3^6*z^12 + x1^17*x2^13*x3^6*z^12 - x1^15*x2^15*x3^6*z^12 - x1^18*x2^11*x3^7*z^12 + x1^16*x2^12*x3^8*z^12 - x1^17*x2^11*x3^5*z^11 + 2*x1^16*x2^12*x3^5*z^11 + x1^16*x2^10*x3^7*z^11 + x1^15*x2^11*x3^7*z^11 + x1^15*x2^11*x3^4*z^10 + x1^14*x2^12*x3^4*z^10 + 2*x1^14*x2^10*x3^6*z^10 - x1^13*x2^11*x3^6*z^10 + x1^14*x2^10*x3^3*z^9 - x1^12*x2^11*x3^4*z^9 - x1^15*x2^7*x3^5*z^9 + x1^13*x2^9*x3^5*z^9 + x1^12*x2^10*x3^5*z^9 + x1^14*x2^7*x3^6*z^9 - x1^12*x2^9*x3^6*z^9 + x1^12*x2^10*x3^2*z

Explanation of process for getting conjectured denominator for B_4 2 variables: ```(1 - x1**4*z) * (1 - x2**4*z) * (1 - x1**2*x2**2*z)**2 * (1 - x1**3*x2*z) * (1 - x1*x2**3*z) * (1 + x1**2*x2**2*z) * (1 - x1**6*x2**6*z**3)```

First try conjectured denominator = 1 

From degree 3 onward we have difference tuples of (2, 2, 0) => (degree of $\nu$ increases by 1 => the exponents for  $x_1$ and $x_2$ increase by exactly 2) 

=> we have a common ratio of $x_1^2 x_2^2 z$ =>

rational factor in denominator:$$(1 - x_1^2 x_2^2 z)$$

Using containment principle we know it will have the denominator from B_4 1 var: (1 - x1**4*z) =>

So we test new conjectured denominator: ```(1 - x1**4*z) * (1 - x1**2*x2**2*z)```

...... 



Will's Contribution B_3 in 2 and 3 variables. I hope this is ok.

In [1]:
def compute_B_mu(mu_part, max_deg_nu, n_vars, m_vars=None):
    Sym = SymmetricFunctions(QQ)
    s = Sym.schur()
    mu = Partition(mu_part)
    results = {}
    
    for d in range(1, max_deg_nu + 1):
        for nu in Partitions(d):
            if m_vars is not None and len(nu) > m_vars:
                continue
                
            plethysm_val = s[nu](s[mu])
            restricted_val = sum(c * s[la] for la, c in plethysm_val if len(la) <= n_vars)
            
            if restricted_val:
                results[nu] = restricted_val
                
    return results

B3_2_vars = compute_B_mu([3], 5, 2)
B3_3_vars = compute_B_mu([3], 5, 3)

In [11]:
B3_2_vars

{[1]: s[3],
 [2]: s[4, 2] + s[6],
 [1, 1]: s[3, 3] + s[5, 1],
 [3]: s[6, 3] + s[7, 2] + s[9],
 [2, 1]: s[5, 4] + s[6, 3] + s[7, 2] + s[8, 1],
 [1, 1, 1]: s[6, 3],
 [4]: s[6, 6] + s[8, 4] + s[9, 3] + s[10, 2] + s[12],
 [3, 1]: 2*s[7, 5] + s[8, 4] + 2*s[9, 3] + s[10, 2] + s[11, 1],
 [2, 2]: s[6, 6] + 2*s[8, 4] + s[10, 2],
 [2, 1, 1]: s[7, 5] + s[8, 4] + s[9, 3],
 [1, 1, 1, 1]: s[6, 6],
 [5]: s[9, 6] + s[10, 5] + s[11, 4] + s[12, 3] + s[13, 2] + s[15],
 [4, 1]: s[8, 7] + 2*s[9, 6] + 2*s[10, 5] + 2*s[11, 4] + 2*s[12, 3] + s[13, 2] + s[14, 1],
 [3, 2]: s[8, 7] + 2*s[9, 6] + 2*s[10, 5] + 2*s[11, 4] + s[12, 3] + s[13, 2],
 [3, 1, 1]: s[8, 7] + s[9, 6] + 2*s[10, 5] + s[11, 4] + s[12, 3],
 [2, 2, 1]: s[8, 7] + s[9, 6] + s[10, 5] + s[11, 4],
 [2, 1, 1, 1]: s[9, 6]}

In [12]:
B3_3_vars

{[1]: s[3],
 [2]: s[4, 2] + s[6],
 [1, 1]: s[3, 3] + s[5, 1],
 [3]: s[4, 4, 1] + s[5, 2, 2] + s[6, 3] + s[7, 2] + s[9],
 [2, 1]: s[4, 3, 2] + s[5, 3, 1] + s[5, 4] + s[6, 2, 1] + s[6, 3] + s[7, 2] + s[8, 1],
 [1, 1, 1]: s[3, 3, 3] + s[5, 3, 1] + s[6, 3] + s[7, 1, 1],
 [4]: s[4, 4, 4] + s[6, 4, 2] + s[6, 6] + s[7, 3, 2] + s[7, 4, 1] + s[8, 2, 2] + s[8, 4] + s[9, 3] + s[10, 2] + s[12],
 [3, 1]: s[5, 4, 3] + s[5, 5, 2] + s[6, 3, 3] + 2*s[6, 4, 2] + s[6, 5, 1] + 2*s[7, 3, 2] + 2*s[7, 4, 1] + 2*s[7, 5] + s[8, 2, 2] + 2*s[8, 3, 1] + s[8, 4] + s[9, 2, 1] + 2*s[9, 3] + s[10, 2] + s[11, 1],
 [2, 2]: s[5, 4, 3] + 2*s[6, 4, 2] + s[6, 5, 1] + s[6, 6] + s[7, 3, 2] + s[7, 4, 1] + s[8, 2, 2] + s[8, 3, 1] + 2*s[8, 4] + s[9, 2, 1] + s[10, 2],
 [2, 1, 1]: s[5, 4, 3] + s[5, 5, 2] + 2*s[6, 3, 3] + s[6, 4, 2] + 2*s[6, 5, 1] + 2*s[7, 3, 2] + 2*s[7, 4, 1] + s[7, 5] + 2*s[8, 3, 1] + s[8, 4] + s[9, 2, 1] + s[9, 3] + s[10, 1, 1],
 [1, 1, 1, 1]: s[6, 3, 3] + s[6, 4, 2] + s[6, 6] + s[7, 4, 1] + s[8, 3, 1],
 [5]: s

In [1]:
def compute_B_4(max_deg_nu, n_vars, m_vars=None):
    Sym = SymmetricFunctions(QQ)
    s = Sym.schur()
    mu = Partition([4])
    results = {}
    
    for d in range(1, max_deg_nu + 1):
        for nu in Partitions(d):
            if m_vars is not None and len(nu) > m_vars:
                continue
                
            plethysm_val = s[nu](s[mu])
            restricted_val = sum(c * s[la] for la, c in plethysm_val if len(la) <= n_vars)
            
            if restricted_val:
                results[nu] = restricted_val
                
    return results



In [2]:
B4_2_vars = compute_B_4(4, 2)

In [3]:
B4_2_vars

{[1]: s[4],
 [2]: s[4, 4] + s[6, 2] + s[8],
 [1, 1]: s[5, 3] + s[7, 1],
 [3]: s[6, 6] + s[8, 4] + s[9, 3] + s[10, 2] + s[12],
 [2, 1]: s[7, 5] + 2*s[8, 4] + s[9, 3] + s[10, 2] + s[11, 1],
 [1, 1, 1]: s[7, 5] + s[9, 3],
 [4]: s[8, 8] + 2*s[10, 6] + 2*s[12, 4] + s[13, 3] + s[14, 2] + s[16],
 [3, 1]: 2*s[9, 7] + 2*s[10, 6] + 3*s[11, 5] + 2*s[12, 4] + 2*s[13, 3] + s[14, 2] + s[15, 1],
 [2, 2]: 2*s[8, 8] + 2*s[10, 6] + s[11, 5] + 2*s[12, 4] + s[14, 2],
 [2, 1, 1]: 2*s[9, 7] + s[10, 6] + 2*s[11, 5] + s[12, 4] + s[13, 3],
 [1, 1, 1, 1]: s[10, 6]}

In [4]:
B4_3_vars = compute_B_4(4, 3)

In [6]:
B4_3_vars

{[1]: s[4],
 [2]: s[4, 4] + s[6, 2] + s[8],
 [1, 1]: s[5, 3] + s[7, 1],
 [3]: s[4, 4, 4] + s[6, 4, 2] + s[6, 6] + s[7, 4, 1] + s[8, 2, 2] + s[8, 4] + s[9, 3] + s[10, 2] + s[12],
 [2, 1]: s[5, 4, 3] + s[6, 4, 2] + s[6, 5, 1] + s[7, 3, 2] + s[7, 4, 1] + s[7, 5] + s[8, 3, 1] + 2*s[8, 4] + s[9, 2, 1] + s[9, 3] + s[10, 2] + s[11, 1],
 [1, 1, 1]: s[5, 5, 2] + s[6, 3, 3] + s[7, 4, 1] + s[7, 5] + s[8, 3, 1] + s[9, 3] + s[10, 1, 1],
 [4]: s[6, 6, 4] + s[7, 6, 3] + 2*s[8, 4, 4] + 2*s[8, 6, 2] + s[8, 8] + s[9, 4, 3] + s[9, 5, 2] + s[9, 6, 1] + 2*s[10, 4, 2] + s[10, 5, 1] + 2*s[10, 6] + s[11, 3, 2] + s[11, 4, 1] + s[12, 2, 2] + 2*s[12, 4] + s[13, 3] + s[14, 2] + s[16],
 [3, 1]: s[6, 6, 4] + 2*s[7, 5, 4] + 2*s[7, 6, 3] + s[7, 7, 2] + 2*s[8, 4, 4] + 3*s[8, 5, 3] + 3*s[8, 6, 2] + 2*s[8, 7, 1] + 3*s[9, 4, 3] + 4*s[9, 5, 2] + 3*s[9, 6, 1] + 2*s[9, 7] + s[10, 3, 3] + 4*s[10, 4, 2] + 3*s[10, 5, 1] + 2*s[10, 6] + 2*s[11, 3, 2] + 3*s[11, 4, 1] + 3*s[11, 5] + s[12, 2, 2] + 2*s[12, 3, 1] + 2*s[12, 4] + s[13,